In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings, os
warnings.filterwarnings("ignore")

In [ ]:
# ============================================================
# 0. LOAD DATA
# ============================================================

df = pd.read_parquet(
    "G:/Mi unidad/Master_LSE/Git-hub/Economic_Observatory/"
    "data/processed_data/analysis_panel.parquet",
    engine="fastparquet"
)
df.columns = df.columns.str.strip().str.lower()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Loaded: 17,150 rows × 57 columns


In [ ]:
df.head(5)

In [17]:
# ============================================================
# SETTINGS
# ============================================================
BOUNDARY_PATH = (
    r"G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory"
    r"\Data\raw_data\boundaries"
    r"\UK_Local_Authority_Districts_December_2023_Boundaries_UK_BGC_2537431731774104276.GeoJSON"
)
OUTPUT_DIR  = r"G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory\Data"
VARS        = ["cagr_bus", "cagr_emp", "lq_bus", "lq_emp"]
GEO_COL     = "LAD23CD"       # shapefile join key
DATA_COL    = "geography_code" # df_cs join key

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [18]:
#============================================================
# 0. PREPARE DATA
# ============================================================
# Cross-section — latest year, England only
END_YEAR = df["year"].max()
df_cs = df[
    df["year"] == END_YEAR
].copy()

# Keep only rows with all four variables present
df_map = df_cs[["geography_code"] + VARS].dropna().copy()
print(f"LADs with complete data: {len(df_map)}")

# Load boundaries — England only
gdf = gpd.read_file(BOUNDARY_PATH)
gdf = gdf[gdf[GEO_COL].str.startswith("E")].copy()
gdf = gdf.to_crs(epsg=27700)  # British National Grid for correct proportions

# Merge
gdf = gdf.merge(df_map, left_on=GEO_COL, right_on=DATA_COL, how="left")

# ── helper: save map ─────────────────────────────────────────
def save_map(fig, name):
    path = os.path.join(OUTPUT_DIR, f"{name}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight",
                facecolor="white")
    plt.close(fig)
    print(f"Saved → {path}")


LADs with complete data: 2040


In [19]:
# ============================================================
# OPTION A — MEDIAN SPLIT TYPOLOGY
# ============================================================
print("\n=== OPTION A: Median split ===")

med = df_map[VARS].median()

gdf["A_bus_growth"] = np.where(gdf["cagr_bus"] >= med["cagr_bus"], "+", "−")
gdf["A_emp_growth"] = np.where(gdf["cagr_emp"] >= med["cagr_emp"], "+", "−")
gdf["A_bus_spec"]   = np.where(gdf["lq_bus"]   >= med["lq_bus"],   "+", "−")
gdf["A_emp_spec"]   = np.where(gdf["lq_emp"]   >= med["lq_emp"],   "+", "−")

def assign_type_a(row):
    g = row["A_bus_growth"] == "+" and row["A_emp_growth"] == "+"
    s = row["A_bus_spec"]   == "+" and row["A_emp_spec"]   == "+"
    bg = row["A_bus_growth"] == "+"
    eg = row["A_emp_growth"] == "+"
    if g and s:     return "1 — High growth + High spec"
    if g and not s: return "2 — High growth + Low spec"
    if not g and s: return "3 — Low growth  + High spec"
    if not g and not s: return "4 — Low growth  + Low spec"
    if bg and not eg:   return "5 — Bus grows, Emp declines"
    return "6 — Mixed"

gdf["type_A"] = gdf.apply(assign_type_a, axis=1)

COLOR_A = {
    "1 — High growth + High spec":  "#08306b",  # darkest
    "2 — High growth + Low spec":   "#2171b5",
    "3 — Low growth  + High spec":  "#6baed6",
    "4 — Low growth  + Low spec":   "#c6dbef",  # lightest
    "5 — Bus grows, Emp declines":  "#9ecae1",
    "6 — Mixed":                    "#deebf7",
    "nan":                          "#f0f0f0",
}

fig, ax = plt.subplots(1, 1, figsize=(10, 12))
for ttype, color in COLOR_A.items():
    subset = gdf[gdf["type_A"] == ttype]
    if len(subset):
        subset.plot(ax=ax, color=color, linewidth=0.2,
                    edgecolor="white")
gdf[gdf["type_A"].isna()].plot(ax=ax, color="#f0f0f0",
                                linewidth=0.2, edgecolor="white")
ax.set_axis_off()
ax.set_title("Option A — Median Split Typology", fontsize=14,
             fontweight="bold", pad=12)
patches = [mpatches.Patch(color=c, label=l)
           for l, c in COLOR_A.items() if l != "nan"]
ax.legend(handles=patches, loc="lower left", fontsize=8,
          framealpha=0.9, title="LAD Type")
save_map(fig, "mapA_median_split")

# Summary table A
summary_a = (gdf.groupby("type_A")
               .size()
               .reset_index(name="n_LADs")
               .sort_values("type_A"))
print(summary_a.to_string(index=False))




=== OPTION A: Median split ===
Saved → G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory\Data\mapA_median_split.png
                     type_A  n_LADs
1 — High growth + High spec     233
 2 — High growth + Low spec     295
3 — Low growth  + High spec     496
 4 — Low growth  + Low spec     712


In [20]:
# ============================================================
# OPTION B — K-MEANS CLUSTERING
# ============================================================
print("\n=== OPTION B: K-means clustering ===")

X_clust = StandardScaler().fit_transform(df_map[VARS].values)

# Find best k by silhouette score
sil_scores = {}
for k in range(3, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_clust)
    sil_scores[k] = round(silhouette_score(X_clust, labels), 4)
    print(f"  k={k} | silhouette={sil_scores[k]}")

best_k = max(sil_scores, key=sil_scores.get)
print(f"\nBest k = {best_k} (silhouette={sil_scores[best_k]})")

km_best = KMeans(n_clusters=best_k, random_state=42, n_init=20)
df_map["cluster_B"] = km_best.fit_predict(X_clust)

# Cluster profiles
profile_b = (df_map.groupby("cluster_B")[VARS]
               .mean()
               .round(4))
print("\nCluster profiles:")
print(profile_b.to_string())

# Merge cluster back
gdf = gdf.merge(
    df_map[["geography_code", "cluster_B"]],
    on="geography_code", how="left"
)

COLORS_B = ["#08306b", "#2171b5", "#4292c6",
            "#6baed6", "#9ecae1", "#c6dbef", "#deebf7"]

fig, ax = plt.subplots(1, 1, figsize=(10, 12))
for i in range(best_k):
    subset = gdf[gdf["cluster_B"] == i]
    subset.plot(ax=ax, color=COLORS_B[i], linewidth=0.2,
                edgecolor="white")
gdf[gdf["cluster_B"].isna()].plot(ax=ax, color="#f0f0f0",
                                   linewidth=0.2, edgecolor="white")
ax.set_axis_off()
ax.set_title(f"Option B — K-means Clustering (k={best_k})",
             fontsize=14, fontweight="bold", pad=12)
patches_b = [mpatches.Patch(color=COLORS_B[i],
             label=f"Cluster {i}") for i in range(best_k)]
ax.legend(handles=patches_b, loc="lower left", fontsize=8,
          framealpha=0.9, title="Cluster")
save_map(fig, "mapB_kmeans")


=== OPTION B: K-means clustering ===
  k=3 | silhouette=0.5993
  k=4 | silhouette=0.5985
  k=5 | silhouette=0.4331
  k=6 | silhouette=0.4049
  k=7 | silhouette=0.4146

Best k = 3 (silhouette=0.5993)

Cluster profiles:
           cagr_bus  cagr_emp  lq_bus  lq_emp
cluster_B                                    
0            0.0212    0.0559  1.7889  2.8786
1           -0.0037    0.0001  0.8048  0.6583
2           -1.0000    0.0356  0.0000  1.2102
Saved → G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory\Data\mapB_kmeans.png


In [15]:
# ============================================================
# OPTION C — QUADRANT SCORING
# ============================================================
print("\n=== OPTION C: Quadrant scoring ===")

# Standardise each variable first
sc = StandardScaler()
scaled = sc.fit_transform(df_map[VARS])
df_map[["z_cagr_bus","z_cagr_emp","z_lq_bus","z_lq_emp"]] = scaled

# Composite scores
df_map["growth_score"] = (df_map["z_cagr_bus"] +
                           df_map["z_cagr_emp"]) / 2
df_map["spec_score"]   = (df_map["z_lq_bus"] +
                           df_map["z_lq_emp"]) / 2

def quadrant(row):
    g = row["growth_score"] >= 0
    s = row["spec_score"]   >= 0
    if g and s:      return "Q1 — Dynamic cluster\n(High growth, High spec)"
    if g and not s:  return "Q2 — Emerging growth\n(High growth, Low spec)"
    if not g and s:  return "Q3 — Mature cluster\n(Low growth, High spec)"
    return           "Q4 — Lagging\n(Low growth, Low spec)"

df_map["quad_C"] = df_map.apply(quadrant, axis=1)

gdf = gdf.merge(
    df_map[["geography_code","growth_score","spec_score","quad_C"]],
    on="geography_code", how="left"
)

COLOR_C = {
    "Q1 — Dynamic cluster\n(High growth, High spec)":  "#08306b",  # darkest
    "Q2 — Emerging growth\n(High growth, Low spec)":   "#2171b5",
    "Q3 — Mature cluster\n(Low growth, High spec)":    "#6baed6",
    "Q4 — Lagging\n(Low growth, Low spec)":            "#c6dbef",  # lightest
}

fig, ax = plt.subplots(1, 1, figsize=(10, 12))
for qtype, color in COLOR_C.items():
    subset = gdf[gdf["quad_C"] == qtype]
    if len(subset):
        subset.plot(ax=ax, color=color, linewidth=0.2,
                    edgecolor="white")
gdf[gdf["quad_C"].isna()].plot(ax=ax, color="#f0f0f0",
                                linewidth=0.2, edgecolor="white")
ax.set_axis_off()
ax.set_title("Option C — Quadrant Scoring", fontsize=14,
             fontweight="bold", pad=12)
patches_c = [mpatches.Patch(color=c, label=l.replace("\n", " "))
             for l, c in COLOR_C.items()]
ax.legend(handles=patches_c, loc="lower left", fontsize=8,
          framealpha=0.9, title="Quadrant")
save_map(fig, "mapC_quadrant")

# Scatter plot of the two composite scores
fig2, ax2 = plt.subplots(figsize=(8, 6))
for qtype, color in COLOR_C.items():
    sub = df_map[df_map["quad_C"] == qtype]
    ax2.scatter(sub["spec_score"], sub["growth_score"],
                c=color, alpha=0.7, s=20, edgecolors="white",
                linewidths=0.3, label=qtype.replace("\n", " "))
ax2.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax2.axvline(0, color="gray", linewidth=0.8, linestyle="--")
ax2.set_xlabel("Specialisation score (lq_bus + lq_emp)", fontsize=11)
ax2.set_ylabel("Growth score (cagr_bus + cagr_emp)", fontsize=11)
ax2.set_title("Option C — LAD quadrant positions", fontsize=13,
              fontweight="bold")
ax2.legend(fontsize=7, loc="upper right")
path_scatter = os.path.join(OUTPUT_DIR, "mapC_scatter.png")
fig2.savefig(path_scatter, dpi=150, bbox_inches="tight",
             facecolor="white")
plt.close(fig2)
print(f"Saved → {path_scatter}")


=== OPTION C: Quadrant scoring ===
Saved → G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory\Data\mapC_quadrant.png
Saved → G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory\Data\mapC_scatter.png


In [ ]:
# ============================================================
# EXPORT SUMMARY TABLES
# ============================================================
summary_b = profile_b.copy()
summary_b["n_LADs"] = df_map.groupby("cluster_B").size()

summary_c = (df_map.groupby("quad_C")[VARS + ["growth_score","spec_score"]]
               .agg(["mean","std","count"])
               .round(4))

excel_path = os.path.join(OUTPUT_DIR, "typology_comparison_v2.xlsx")
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    summary_a.to_excel(writer, sheet_name="A_median_split",  index=False)
    summary_b.to_excel(writer, sheet_name="B_kmeans_profile")
    summary_c.to_excel(writer, sheet_name="C_quadrant_profile")
    pd.DataFrame(sil_scores.items(),
                 columns=["k","silhouette"]).to_excel(
        writer, sheet_name="B_silhouette_scores", index=False)

print(f"\nAll done. Excel → {excel_path}")
print("Maps saved to:", OUTPUT_DIR)

In [5]:

# ============================================================
# 0. PREPARE DATA
# ============================================================
# Cross-section — latest year, England only
END_YEAR = df["year"].max()
df_cs = df[
    df["geography_code"].str.startswith("E") &
    (df["year"] == END_YEAR)
].copy()

# Keep only rows with all four variables present
df_map = df_cs[["geography_code"] + VARS].dropna().copy()
print(f"LADs with complete data: {len(df_map)}")

# Load boundaries — England only
gdf = gpd.read_file(BOUNDARY_PATH)
gdf = gdf[gdf[GEO_COL].str.startswith("E")].copy()
gdf = gdf.to_crs(epsg=27700)  # British National Grid for correct proportions

# Merge
gdf = gdf.merge(df_map, left_on=GEO_COL, right_on=DATA_COL, how="left")

# ── helper: save map ─────────────────────────────────────────
def save_map(fig, name):
    path = os.path.join(OUTPUT_DIR, f"{name}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight",
                facecolor="white")
    plt.close(fig)
    print(f"Saved → {path}")


# ============================================================
# OPTION A — MEDIAN SPLIT TYPOLOGY
# ============================================================
print("\n=== OPTION A: Median split ===")

med = df_map[VARS].median()

gdf["A_bus_growth"] = np.where(gdf["cagr_bus"] >= med["cagr_bus"], "+", "−")
gdf["A_emp_growth"] = np.where(gdf["cagr_emp"] >= med["cagr_emp"], "+", "−")
gdf["A_bus_spec"]   = np.where(gdf["lq_bus"]   >= med["lq_bus"],   "+", "−")
gdf["A_emp_spec"]   = np.where(gdf["lq_emp"]   >= med["lq_emp"],   "+", "−")

def assign_type_a(row):
    g = row["A_bus_growth"] == "+" and row["A_emp_growth"] == "+"
    s = row["A_bus_spec"]   == "+" and row["A_emp_spec"]   == "+"
    bg = row["A_bus_growth"] == "+"
    eg = row["A_emp_growth"] == "+"
    if g and s:     return "1 — High growth + High spec"
    if g and not s: return "2 — High growth + Low spec"
    if not g and s: return "3 — Low growth  + High spec"
    if not g and not s: return "4 — Low growth  + Low spec"
    if bg and not eg:   return "5 — Bus grows, Emp declines"
    return "6 — Mixed"

gdf["type_A"] = gdf.apply(assign_type_a, axis=1)

COLOR_A = {
    "1 — High growth + High spec":  "#2d6a4f",
    "2 — High growth + Low spec":   "#74c69d",
    "3 — Low growth  + High spec":  "#f4a261",
    "4 — Low growth  + Low spec":   "#e63946",
    "5 — Bus grows, Emp declines":  "#457b9d",
    "6 — Mixed":                    "#adb5bd",
    "nan":                          "#f0f0f0",
}

fig, ax = plt.subplots(1, 1, figsize=(10, 12))
for ttype, color in COLOR_A.items():
    subset = gdf[gdf["type_A"] == ttype]
    if len(subset):
        subset.plot(ax=ax, color=color, linewidth=0.2,
                    edgecolor="white")
gdf[gdf["type_A"].isna()].plot(ax=ax, color="#f0f0f0",
                                linewidth=0.2, edgecolor="white")
ax.set_axis_off()
ax.set_title("Option A — Median Split Typology", fontsize=14,
             fontweight="bold", pad=12)
patches = [mpatches.Patch(color=c, label=l)
           for l, c in COLOR_A.items() if l != "nan"]
ax.legend(handles=patches, loc="lower left", fontsize=8,
          framealpha=0.9, title="LAD Type")
save_map(fig, "mapA_median_split")

# Summary table A
summary_a = (gdf.groupby("type_A")
               .size()
               .reset_index(name="n_LADs")
               .sort_values("type_A"))
print(summary_a.to_string(index=False))


# ============================================================
# OPTION B — K-MEANS CLUSTERING
# ============================================================
print("\n=== OPTION B: K-means clustering ===")

X_clust = StandardScaler().fit_transform(df_map[VARS].values)

# Find best k by silhouette score
sil_scores = {}
for k in range(3, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_clust)
    sil_scores[k] = round(silhouette_score(X_clust, labels), 4)
    print(f"  k={k} | silhouette={sil_scores[k]}")

best_k = max(sil_scores, key=sil_scores.get)
print(f"\nBest k = {best_k} (silhouette={sil_scores[best_k]})")

km_best = KMeans(n_clusters=best_k, random_state=42, n_init=20)
df_map["cluster_B"] = km_best.fit_predict(X_clust)

# Cluster profiles
profile_b = (df_map.groupby("cluster_B")[VARS]
               .mean()
               .round(4))
print("\nCluster profiles:")
print(profile_b.to_string())

# Merge cluster back
gdf = gdf.merge(
    df_map[["geography_code", "cluster_B"]],
    on="geography_code", how="left"
)

COLORS_B = ["#2d6a4f", "#74c69d", "#f4a261",
            "#e63946", "#457b9d", "#9b5de5", "#adb5bd"]

fig, ax = plt.subplots(1, 1, figsize=(10, 12))
for i in range(best_k):
    subset = gdf[gdf["cluster_B"] == i]
    subset.plot(ax=ax, color=COLORS_B[i], linewidth=0.2,
                edgecolor="white")
gdf[gdf["cluster_B"].isna()].plot(ax=ax, color="#f0f0f0",
                                   linewidth=0.2, edgecolor="white")
ax.set_axis_off()
ax.set_title(f"Option B — K-means Clustering (k={best_k})",
             fontsize=14, fontweight="bold", pad=12)
patches_b = [mpatches.Patch(color=COLORS_B[i],
             label=f"Cluster {i}") for i in range(best_k)]
ax.legend(handles=patches_b, loc="lower left", fontsize=8,
          framealpha=0.9, title="Cluster")
save_map(fig, "mapB_kmeans")


# ============================================================
# OPTION C — QUADRANT SCORING
# ============================================================
print("\n=== OPTION C: Quadrant scoring ===")

# Standardise each variable first
sc = StandardScaler()
scaled = sc.fit_transform(df_map[VARS])
df_map[["z_cagr_bus","z_cagr_emp","z_lq_bus","z_lq_emp"]] = scaled

# Composite scores
df_map["growth_score"] = (df_map["z_cagr_bus"] +
                           df_map["z_cagr_emp"]) / 2
df_map["spec_score"]   = (df_map["z_lq_bus"] +
                           df_map["z_lq_emp"]) / 2

def quadrant(row):
    g = row["growth_score"] >= 0
    s = row["spec_score"]   >= 0
    if g and s:      return "Q1 — Dynamic cluster\n(High growth, High spec)"
    if g and not s:  return "Q2 — Emerging growth\n(High growth, Low spec)"
    if not g and s:  return "Q3 — Mature cluster\n(Low growth, High spec)"
    return           "Q4 — Lagging\n(Low growth, Low spec)"

df_map["quad_C"] = df_map.apply(quadrant, axis=1)

gdf = gdf.merge(
    df_map[["geography_code","growth_score","spec_score","quad_C"]],
    on="geography_code", how="left"
)

COLOR_C = {
    "Q1 — Dynamic cluster\n(High growth, High spec)":  "#2d6a4f",
    "Q2 — Emerging growth\n(High growth, Low spec)":   "#74c69d",
    "Q3 — Mature cluster\n(Low growth, High spec)":    "#f4a261",
    "Q4 — Lagging\n(Low growth, Low spec)":            "#e63946",
}

fig, ax = plt.subplots(1, 1, figsize=(10, 12))
for qtype, color in COLOR_C.items():
    subset = gdf[gdf["quad_C"] == qtype]
    if len(subset):
        subset.plot(ax=ax, color=color, linewidth=0.2,
                    edgecolor="white")
gdf[gdf["quad_C"].isna()].plot(ax=ax, color="#f0f0f0",
                                linewidth=0.2, edgecolor="white")
ax.set_axis_off()
ax.set_title("Option C — Quadrant Scoring", fontsize=14,
             fontweight="bold", pad=12)
patches_c = [mpatches.Patch(color=c, label=l.replace("\n", " "))
             for l, c in COLOR_C.items()]
ax.legend(handles=patches_c, loc="lower left", fontsize=8,
          framealpha=0.9, title="Quadrant")
save_map(fig, "mapC_quadrant")

# Scatter plot of the two composite scores
fig2, ax2 = plt.subplots(figsize=(8, 6))
for qtype, color in COLOR_C.items():
    sub = df_map[df_map["quad_C"] == qtype]
    ax2.scatter(sub["spec_score"], sub["growth_score"],
                c=color, alpha=0.6, s=20,
                label=qtype.replace("\n", " "))
ax2.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax2.axvline(0, color="gray", linewidth=0.8, linestyle="--")
ax2.set_xlabel("Specialisation score (lq_bus + lq_emp)", fontsize=11)
ax2.set_ylabel("Growth score (cagr_bus + cagr_emp)", fontsize=11)
ax2.set_title("Option C — LAD quadrant positions", fontsize=13,
              fontweight="bold")
ax2.legend(fontsize=7, loc="upper right")
path_scatter = os.path.join(OUTPUT_DIR, "mapC_scatter.png")
fig2.savefig(path_scatter, dpi=150, bbox_inches="tight",
             facecolor="white")
plt.close(fig2)
print(f"Saved → {path_scatter}")

# ============================================================
# EXPORT SUMMARY TABLES
# ============================================================
summary_b = profile_b.copy()
summary_b["n_LADs"] = df_map.groupby("cluster_B").size()

summary_c = (df_map.groupby("quad_C")[VARS + ["growth_score","spec_score"]]
               .agg(["mean","std","count"])
               .round(4))

excel_path = os.path.join(OUTPUT_DIR, "typology_comparison.xlsx")
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    summary_a.to_excel(writer, sheet_name="A_median_split",  index=False)
    summary_b.to_excel(writer, sheet_name="B_kmeans_profile")
    summary_c.to_excel(writer, sheet_name="C_quadrant_profile")
    pd.DataFrame(sil_scores.items(),
                 columns=["k","silhouette"]).to_excel(
        writer, sheet_name="B_silhouette_scores", index=False)

print(f"\nAll done. Excel → {excel_path}")
print("Maps saved to:", OUTPUT_DIR)

LADs with complete data: 1736

=== OPTION A: Median split ===
Saved → G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory\Data\mapA_median_split.png
                     type_A  n_LADs
1 — High growth + High spec     207
 2 — High growth + Low spec     296
3 — Low growth  + High spec     465
 4 — Low growth  + Low spec     768

=== OPTION B: K-means clustering ===
  k=3 | silhouette=0.5882
  k=4 | silhouette=0.5919
  k=5 | silhouette=0.4774
  k=6 | silhouette=0.4272
  k=7 | silhouette=0.3014

Best k = 4 (silhouette=0.5919)

Cluster profiles:
           cagr_bus  cagr_emp  lq_bus  lq_emp
cluster_B                                    
0            0.0175    0.0420  1.8158  2.8579
1           -0.0023    0.0005  0.8337  0.6726
2           -0.5000    2.7361  0.2937  0.7548
3           -1.0000   -0.0824  0.0000  1.0435
Saved → G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory\Data\mapB_kmeans.png

=== OPTION C: Quadrant scoring ===
Saved → G:\Mi unidad\Master_LSE\Git-hub\Economic_Observat